<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/10_Save_Load_TransferLearnign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this section we will look at how to persist model state with saving, loading and running model predictions.

In [2]:
import torch
import torchvision.models as models

# Saving and Loading Model Weights

PyTorch models store the learned parameters in an internal state dictionary, called state_dict. These can be persisted via the torch.save method:

In [3]:
model = models.vgg16(weights='IMAGENET1K_V1')
# model = models.vgg16(weights='IMAGENET1K_V1'): carica il modello VGG16 pre-addestrato
# - models.vgg16: modello VGG16 (architettura di rete convoluzionale profonda)
# - weights='IMAGENET1K_V1': specifica di caricare i pesi pre-addestrati su ImageNet
#   * ImageNet: dataset con 1.2 milioni di immagini, 1000 classi
#   * 'IMAGENET1K_V1' è la versione dei pesi
#   * Il modello ha già "imparato" a riconoscere caratteristiche generali delle immagini
# - Questo è il TRANSFER LEARNING: usare un modello già addestrato come punto di partenza

# COSA CONTIENE model ORA:
# - Architettura VGG16 completa (13 layer convoluzionali + 3 fully connected)
# - Pesi già ottimizzati per riconoscere bordi, texture, forme, oggetti
# - Layer finale: 1000 neuroni (uno per ogni classe di ImageNet)

torch.save(model.state_dict(), 'model_weights.pth')
# torch.save(model.state_dict(), 'model_weights.pth'): SALVA SOLO I PESI del modello
# - model.state_dict(): dizionario Python che contiene:
#   * Per ogni layer, i pesi (weight) e i bias come tensor
#   * Mappa: 'layer_name.weight' → tensor, 'layer_name.bias' → tensor
#   * Non salva l'architettura, solo i numeri (parametri)
# - 'model_weights.pth': nome del file dove salvare
#   * .pth è l'estensione convenzionale per PyTorch (ma potrebbe essere .pt)
#   * Il file viene salvato nella directory corrente

# PER CARICARE I PESI IN UN SECONDO MOMENTO:
# model = models.vgg16()  # crea modello SENZA pesi (architettura vuota)
# model.load_state_dict(torch.load('model_weights.pth'))  # carica i pesi salvati
# model.eval()  # imposta modalità valutazione

# DIFFERENZA TRA SALVATAGGIO COMPLETO E SOLO STATO:

# 1. SALVATAGGIO COMPLETO (modello + pesi):
torch.save(model, 'model_complete.pth')
# - Salva TUTTO: architettura, pesi, ottimizzatore, ecc.
# - Poi puoi caricare con: model = torch.load('model_complete.pth')
# - ATTENZIONE: meno flessibile, dipende dalla struttura esatta del codice

# 2. SOLO STATO (solo pesi) - COME NEL TUO CODICE:
torch.save(model.state_dict(), 'model_weights.pth')
# - Salva SOLO i parametri numerici
# - Devi ricreare l'architettura prima di caricare i pesi
# - PIÙ FLESSIBILE: puoi usare gli stessi pesi su architetture diverse (se compatibili)

# PERCHÉ USARE MODELLI PRE-ADDESTRATI:
# - Non devi addestrare da zero (risparmia tempo, giorni → ore)
# - Il modello ha già imparato caratteristiche generali (bordi, texture)
# - Funziona bene anche con pochi dati (fine-tuning)
# - Standard per competizioni e applicazioni reali



# ESEMPIO DI FINE-TUNING (adattare a nuovo dataset):
# 1. Carica VGG16 pre-addestrato

# model = models.vgg16(weights='IMAGENET1K_V1')


# 2. Congela layer convoluzionali (opzionale)

# for param in model.features.parameters():
#    param.requires_grad = False


# 3. Sostituisci il classificatore per il tuo numero di classi

# model.classifier[6] = nn.Linear(4096, num_classes)  # es. 10 per Fashion-MNIST


# 4. Addestra solo i nuovi layer (o tutti con learning rate basso)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 77.7MB/s]


To load model weights, you need to create an instance of the same model first, and then load the parameters using load_state_dict() method.

In the code below, we set weights_only=True to limit the functions executed during unpickling to only those necessary for loading weights. Using weights_only=True is considered a best practice when loading weights.

In [5]:
model = models.vgg16()
# model = models.vgg16(): crea un modello VGG16 SENZA pesi pre-addestrati
# - models.vgg16: architettura del modello VGG16
# - Non specifichiamo il parametro "weights", quindi il modello è "vergine"
# - I pesi sono inizializzati casualmente (default di PyTorch)
# - L'architettura è identica al modello pre-addestrato, ma i valori sono diversi

# PERCHÉ NON SPECIFICARE weights?
# - Vogliamo caricare i nostri pesi salvati in precedenza
# - Serve un modello con la STESSA ARCHITETTURA di quello che ha salvato i pesi
# - Se specificassimo weights, caricherebbe altri pesi (ImageNet) che sovrascriveremmo

model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
# model.load_state_dict(...): carica i pesi salvati nel modello
# - torch.load('model_weights.pth'): legge il file salvato in precedenza
#   * 'model_weights.pth' è il file che abbiamo salvato con torch.save()
#   * Contiene un dizionario con i pesi di tutti i layer
#
# - weights_only=True: parametro di SICUREZZA (introdotto in PyTorch recenti)
#   * Carica SOLO i pesi, non esegue codice arbitrario
#   * Protegge da file potenzialmente dannosi
#   * Più sicuro, specialmente se il file viene da fonte esterna
#
# - model.load_state_dict(...): inserisce i pesi letti nel modello
#   * Confronta il dizionario caricato con la struttura del modello
#   * Assegna ogni peso al layer corrispondente
#   * Solleva errore se l'architettura non corrisponde (es. layer mancanti)

model.eval()
# model.eval(): imposta il modello in MODALITÀ VALUTAZIONE
# - Disattiva layer che si comportano diversamente durante training:
#   * Dropout: viene disattivato (non "spegne" neuroni casualmente)
#   * BatchNorm: usa statistiche accumulate (non quelle del batch corrente)
# - IMPORTANTE: fondamentale per ottenere predizioni corrette!
# - Se non chiami eval(), il modello potrebbe dare risultati inconsistenti
# - Di solito si chiama PRIMA di fare inferenza (predizioni)


# FLUSSO COMPLETO DEL CARICAMENTO:

# 1. PRIMA (tempo 0): abbiamo salvato i pesi con:
#    torch.save(model_addestrato.state_dict(), 'model_weights.pth')
#
# 2. ORA (tempo 1): vogliamo riutilizzare quei pesi
#    - Creiamo un modello nuovo con la stessa architettura: model = models.vgg16()
#    - Carichiamo i pesi: model.load_state_dict(torch.load('model_weights.pth'))
#    - Pronto per l'uso: model.eval()
#
# 3. Il modello ora è IDENTICO a quello che avevamo salvato



# PREDIZIONE CON MODELLO CARICATO:


# with torch.no_grad():  # niente gradienti per inferenza
    # Prepara un'immagine (deve essere preprocessata come per ImageNet)
    # output = model(immagine)
    # predizione = output.argmax(1)

# PERCHÉ QUESTO È UTILE:
# - Non devi riaddestrare il modello ogni volta
# - Puoi addestrare su computer potente (GPU), poi usare su computer normale
# - Condividere modelli addestrati con altri
# - Fare esperimenti ripartendo da punti di controllo (checkpoints)

"\n1. PRIMA (tempo 0): abbiamo salvato i pesi con:\n   torch.save(model_addestrato.state_dict(), 'model_weights.pth')\n\n2. ORA (tempo 1): vogliamo riutilizzare quei pesi\n   - Creiamo un modello nuovo con la stessa architettura: model = models.vgg16()\n   - Carichiamo i pesi: model.load_state_dict(torch.load('model_weights.pth'))\n   - Pronto per l'uso: model.eval()\n\n3. Il modello ora è IDENTICO a quello che avevamo salvato\n"

be sure to call model.eval() method before inferencing to set the dropout and batch normalization layers to evaluation mode. Failing to do this will yield inconsistent inference results.

# Saving and Loading Models with Shapes

When loading model weights, we needed to instantiate the model class first, because the class defines the structure of a network. We might want to save the structure of this class together with the model, in which case we can pass model (and not model.state_dict()) to the saving function:

In [7]:
torch.save(model, 'model.pth')
# torch.save(model, 'model.pth'): salva l'INTERO modello (non solo i pesi)
# - model: l'oggetto modello completo (architettura + pesi + eventuale altro stato)
# - 'model.pth': nome del file dove salvare

# COSA CONTIENE QUESTO SALVATAGGIO (MOLTO DI PIÙ del solo state_dict):
# 1. L'ARCHITETTURA DEL MODELLO (la classe, i layer, la struttura)
# 2. I PESI di tutti i layer (parametri addestrati)
# 3. Altri attributi del modello (se presenti)

# DIFFERENZA FONDAMENTALE CON torch.save(model.state_dict()):

# ┌───────────────────────────┬────────────────────────────┬─────────────────────────────────┐
# │                           │ model.state_dict()         │ model (intero)                  │
# ├───────────────────────────┼────────────────────────────┼─────────────────────────────────┤
# │ Cosa salva                │ Solo i pesi (dizionario)   │ Modello COMPLETO                │
# │ Dimensione file           │ Più piccola                │ Più grande                      │
# │ Contiene architettura?    │ NO                         │ SÌ                              │
# │ Per ricaricare serve      │ Ricreare l'architettura    │ No, tutto incluso               │
# │ Flessibilità              │ ALTA (cambia architettura) │ BASSA (deve essere uguale)      │
# │ Sicurezza                 │ Più sicuro (solo numeri)   │ Rischioso (può eseguire codice) │
# │ Compatibilità versioni    │ Più robusta                │ Può rompersi tra versioni       │
# └───────────────────────────┴────────────────────────────┴─────────────────────────────────┘


# PER RICARICARE IL MODELLO SALVATO:

#   model_caricato = torch.load('model.pth')

# - torch.load('model.pth'): carica l'INTERO oggetto modello
# - model_caricato è già pronto all'uso (ha architettura e pesi)
# - model_caricato.eval() per modalità valutazione

# VANTAGGI DEL SALVATAGGIO COMPLETO:
# - Semplicità: una riga per salvare, una per caricare
# - Non devi ricordare l'architettura esatta
# - Comodo per prototipazione rapida
# - Salva anche altri attributi custom del modello

# SVANTAGGI:
# - MENO FLESSIBILE: se cambi l'architettura della classe, non puoi caricare
# - PROBLEMI DI VERSIONI: tra versioni diverse di PyTorch può non funzionare
# - RISCHIO SICUREZZA: il file potrebbe contenere codice malevolo (pickle)
# - DIPENDENZA DAL CODICE: deve essere disponibile la definizione della classe





# # ESEMPIO CONCRETO:
# # 1. Definiamo e addestriamo un modello
# class MioModello(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.fc = nn.Linear(784, 10)
#
#     def forward(self, x):
#         return self.fc(x)
#
# model = MioModello()
# # ... addestramento ...
#
# # 2. Salviamo l'INTERO modello
# torch.save(model, 'mio_modello_completo.pth')
#
# # 3. In un altro script, possiamo caricarlo SENZA dover reimportare la classe?
# # ATTENZIONE: Serve comunque la definizione della classe!
# model_caricato = torch.load('mio_modello_completo.pth')
# # Funziona SOLO se la classe MioModello è definita nello script corrente!
#
# # QUESTO È IL PROBLEMA PRINCIPALE:
# # Il file .pth contiene riferimento alla classe, ma non la definizione
# # Devi avere la stessa classe definita nel contesto di caricamento
#
# # QUANDO USARE L'UNO O L'ALTRO:
# '''
# - Usa model.state_dict() QUANDO:
#   * Vuoi massima flessibilità
#   * Distribuisci modelli ad altri
#   * Cambi spesso architettura
#   * Produzione (best practice)
#
# - Usa model (intero) QUANDO:
#   * Prototipazione veloce
#   * Checkpoint temporanei durante training
#   * Stai lavorando da solo nello stesso ambiente
#   * Vuoi salvare anche ottimizzatore, epoca, ecc. (per riprendere training)
# '''
#
# # PER SALVARE TUTTO (modello + ottimizzatore + epoca) per riprendere training:
# torch.save({
#     'epoch': epoch,
#     'model_state_dict': model.state_dict(),
#     'optimizer_state_dict': optimizer.state_dict(),
#     'loss': loss,
# }, 'checkpoint.pth')
#
# # Poi per riprendere:
# checkpoint = torch.load('checkpoint.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# epoch = checkpoint['epoch']





"\n┌───────────────────────────┬────────────────────────────┬───────────────────────────┐\n│                           │ model.state_dict()         │ model (intero)            │\n├───────────────────────────┼────────────────────────────┼───────────────────────────┤\n│ Cosa salva                │ Solo i pesi (dizionario)   │ Modello COMPLETO          │\n│ Dimensione file           │ Più piccola                 │ Più grande                │\n│ Contiene architettura?    │ NO                          │ SÌ                        │\n│ Per ricaricare serve      │ Ricreare l'architettura    │ No, tutto incluso         │\n│ Flessibilità              │ ALTA (cambia architettura) │ BASSA (deve essere uguale)│\n│ Sicurezza                 │ Più sicuro (solo numeri)   │ Rischioso (può eseguire codice)│\n│ Compatibilità versioni    │ Più robusta                │ Può rompersi tra versioni │\n└───────────────────────────┴────────────────────────────┴───────────────────────────┘\n"

We can then load the model as demonstrated below.

As described in Saving and loading torch.nn.Modules, saving state_dict is considered the best practice. However, below we use weights_only=False because this involves loading the model, which is a legacy use case for torch.save.

In [9]:
model = torch.load('model.pth', weights_only=False)
# model = torch.load('model.pth', weights_only=False): carica un modello salvato precedentemente



# ANALIZZIAMO I COMPONENTI:

# 1. torch.load('model.pth')
#    - Carica il file salvato con torch.save()
#    - 'model.pth' è il nome del file (può essere .pt, .pth, .pkl)

# 2. weights_only=False (IMPORTANTE!)
#    - Parametro di SICUREZZA introdotto in PyTorch recenti
#    - weights_only=False significa: CARICA TUTTO, non solo i pesi
#    - Permette di caricare l'INTERO oggetto modello (architettura + pesi)

# COSA SUCCEDE CON weights_only=False:
# - PyTorch carica l'oggetto completo usando il modulo pickle di Python
# - Ricostruisce l'intero modello con la sua architettura
# - Esegue il codice necessario per ricreare gli oggetti

# RISCHI DI SICUREZZA:
# ┌────────────────────┬────────────────────────────┬───────────────────────────┐
# │ weights_only=True  │ weights_only=False         │ Perché è importante       │
# ├────────────────────┼────────────────────────────┼───────────────────────────┤
# │ Carica SOLO pesi   │ Carica OGGETTI COMPLETI    │ pickle può eseguire       │
# │ (tensor numerici)  │ (usa pickle)               │ codice arbitrario!        │
# │                    │                            │                           │
# │ SICURO             │ POTENZIALMENTE PERICOLOSO  │ File malevolo potrebbe    │
# │                    │                            │ eseguire codice dannoso   │
# │                    │                            │                           │
# │ Consigliato per    │ Usa solo se ti fidi        │ Come aprire un allegato   │
# │ file da fonti      │ della fonte (tuo file)     │ email da uno sconosciuto  │
# │ non fidate         │                            │                           │
# └────────────────────┴────────────────────────────┴───────────────────────────┘

# QUANDO USARE weights_only=False:
# - Stai caricando un modello che HAI SALVATO TU
# - Sei in un ambiente controllato e sicuro
# - Hai bisogno di caricare l'architettura completa
# - Stai riprendendo un training (con ottimizzatore, epoca, ecc.)

# QUANDO EVITARE (usare weights_only=True):
# - Scarichi modelli da Internet (da fonti non verificate)
# - Carichi file creati da altri
# - In ambienti di produzione dove la sicurezza è critica
# - Non sei sicuro della provenienza del file


# # ESEMPIO DI CARICAMENTO SICURO (consigliato per modelli esterni):
# model = torch.load('model_from_internet.pth', weights_only=True)
# # Ma questo funziona SOLO se il file contiene solo pesi (state_dict)
# # Se il file contiene l'intero modello, weights_only=True darà errore!
#
# # ERRORE COMUNE:
# # model = torch.load('model.pth', weights_only=True)
# # Se 'model.pth' contiene l'intero modello (non solo state_dict):
# # RuntimeError: You can't safely load when weights_only=True...
#
# # CARICAMENTO COMPLETO CON GESTIONE DELL'ERRORE:
# try:
#     # Prima proviamo con weights_only=True (sicuro)
#     model = torch.load('model.pth', weights_only=True)
#     print("Caricato come state_dict (solo pesi)")
# except:
#     # Se fallisce, allora carichiamo l'intero modello
#     model = torch.load('model.pth', weights_only=False)
#     print("Caricato come modello completo (potenziale rischio)")
#
# # COSA OTTIENI DOPO QUESTO CARICAMENTO:
# # - model è un oggetto modello completo e funzionante
# # - Ha già l'architettura e i pesi caricati
# # - Puoi usarlo subito per predizioni (dopo .eval())
# # - Oppure continuare l'addestramento
#
# # ESEMPIO COMPLETO:
# # Salvare
# model = models.vgg16(weights='IMAGENET1K_V1')
# torch.save(model, 'vgg16_completo.pth')
#
# # Caricare (ATTENZIONE: weights_only=False necessario!)
# vgg_caricato = torch.load('vgg16_completo.pth', weights_only=False)
# vgg_caricato.eval()  # modalità valutazione
#
# # Ora puoi fare predizioni
# with torch.no_grad():
#     output = vgg_caricato(immagine)





This approach uses Python pickle module when serializing the model, thus it relies on the actual class definition to be available when loading the model.